In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import os

os.chdir('../..')

In [2]:
Q = """
SELECT 
    s.id,
    s.street,
    s.geom,
    EXTRACT(HOUR FROM inst.t) AS hour_of_day,
    AVG(inst.v) AS avg_bikes
FROM station_status_mdb mdb
JOIN stations s ON mdb.station_id = s.id,
LATERAL unnest(instants(mdb.bikes_history)) AS inst(v, t)
GROUP BY s.id, s.street, s.geom, hour_of_day
ORDER BY hour_of_day, s.id;
"""

In [9]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt

from src.utils.database_connection import DatabaseConnector

def plot_hourly_occupancy(hour_to_plot):
    db = DatabaseConnector()
    
    query = """
    SELECT 
        s.id,
        s.geom,
        EXTRACT(HOUR FROM getTimestamp(inst)) AS hour,
        AVG(getValue(inst)) AS avg_bikes
    FROM station_status_mdb mdb
    JOIN stations s ON mdb.station_id = s.id,
    LATERAL unnest(instants(mdb.bikes_history)) AS inst
    WHERE EXTRACT(HOUR FROM getTimestamp(inst)) = %s
    GROUP BY s.id, s.geom, hour;
    """
    
    try:
        results = db.execute_query(query, params=(hour_to_plot,), fetch=True)
        df = pd.DataFrame(results)
        
        # Convert WKB/Hex to Shapely geometry
        df['geometry'] = df['geom'].apply(lambda x: wkt.loads(x) if isinstance(x, str) else x)
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")
        
        fig, ax = plt.subplots(1, 1, figsize=(10, 10))
        gdf.plot(column='avg_bikes', ax=ax, legend=True, cmap='RdYlGn', markersize=50)
        
        plt.title(f"Average Bike Availability at {hour_to_plot}:00")
        plt.show()
        print(f"Plot generated for hour: {hour_to_plot}")
        
    except Exception as e:
        print(f"Failed to plot data: {e}")
    finally:
        db.disconnect()

In [10]:
plot_hourly_occupancy(8)

Database connection established successfully.
Failed to plot data: ParseException: Unknown type: '0101000020E6100000F68178EBE959014043190C5520B24440'
Database connection closed.
